# Model Evaluation Results Viewer

This notebook loads the results of model evaluation runs from `tmp/outputs/model_check` and visualizes the key performance metrics.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

In [2]:
import sys
from pathlib import Path  # noqa: F811

# Resolve root directory path
current_dir = Path.cwd()
if current_dir.name == "eval_models_with_diff_options":
    root_dir = current_dir.parents[1]
else:
    root_dir = current_dir

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(root_dir.parent) not in sys.path:
    sys.path.append(str(root_dir.parent))

from helpers.df_helper import ModelStatsDisplayCols  # noqa: E402

from tools.eval_models_with_diff_options.run_all_models import RESULTS_DIR  # noqa: E402

csv_files = list(RESULTS_DIR.glob("model_comparison_*.csv"))

dfs = []
for csv_file in csv_files:
    # Extract run name from model_comparison_{run_name}.csv
    run_name = csv_file.stem.replace("model_comparison_", "")
    df_run = pd.read_csv(csv_file)
    df_run["Run Name"] = run_name
    dfs.append(df_run)

df_all = pd.concat(dfs, ignore_index=True)

# Define consistent color palette for models dynamically supporting any number of models
unique_models = sorted(df_all[ModelStatsDisplayCols.MODEL].unique())
print(f"Unique models: {len(unique_models)}")
print(f"Loaded {len(dfs)} run results.")

display(df_all.head())

Unique models: 1
Loaded 420 run results.


,Model,Total Time (s),Load Time (s),Prompt Tokens,Gen Tokens,Gen Speed (t/s),Response Chars,Response Words,GPU Usage,GPU Info,Options,Run Name
0,gemma4:e2b-it-qat,68.348741,4.514795,8753,2872,47.453615,9322,1240,0.999981,1.58 GB / 1.58 GB,"ctx: 12288, pred: -2, temp: 0.3",m1_ctx12k_temp03_predneg2
1,gemma4:e2b-it-qat,51.324808,4.807390,8753,2048,47.415860,6030,795,1.000000,1.58 GB / 1.58 GB,"ctx: 16384, pred: 2048, temp: 0.1",m1_ctx16k_temp01_pred2k
2,gemma4:e2b-it-qat,68.836172,4.316595,8753,2939,48.070804,9372,1234,1.000000,1.58 GB / 1.58 GB,"ctx: 12288, pred: 7168, temp: 1.0",m1_ctx12k_temp10_pred7k
3,gemma4:e2b-it-qat,6.559260,4.283859,6143,1,0.000000,2,1,1.000000,1.56 GB / 1.56 GB,"ctx: 6144, pred: 6144, temp: 0.4",m1_ctx6k_temp04_pred6k
4,gemma4:e2b-it-qat,62.959326,4.303841,8753,2694,48.632367,9119,1167,0.999979,1.58 GB / 1.58 GB,"ctx: 14336, pred: -2, temp: 0.0",m1_ctx14k_temp00_predneg2


In [3]:
pd.DataFrame(df_all["Run Name"].unique(), columns=["Run Name"])

,Run Name
0,m1_ctx12k_temp03_predneg2
1,m1_ctx16k_temp01_pred2k
2,m1_ctx12k_temp10_pred7k
3,m1_ctx6k_temp04_pred6k
4,m1_ctx14k_temp00_predneg2
...,...
415,m1_ctx12k_temp01_pred4k
416,m1_ctx16k_temp01_predneg2
417,m1_ctx6k_temp01_predneg1
418,m1_ctx14k_temp01_pred3k


In [4]:
df_all = df_all.round(3)
non_100_gpu = (
    df_all[df_all[ModelStatsDisplayCols.GPU_USAGE] != 1.0]
    .sort_values([ModelStatsDisplayCols.GPU_USAGE])
    .reset_index(drop=True)
)
if not non_100_gpu.empty:
    print(f"Rows with bad GPU utilization: {non_100_gpu.shape[0]}")
    display(
        non_100_gpu[
            [ModelStatsDisplayCols.MODEL, ModelStatsDisplayCols.GPU_USAGE, ModelStatsDisplayCols.GPU_INFO, ModelStatsDisplayCols.OPTIONS, "Run Name"]
        ]
    )
else:
    print("All models ran at 100% GPU Usage.")

Rows with bad GPU utilization: 11


,Model,GPU Usage,GPU Info,Options,Run Name
0,gemma4:e2b-it-qat,0.984,1.56 GB / 1.56 GB,"ctx: 6144, pred: -2, temp: 0.2",m1_ctx6k_temp02_predneg2
1,gemma4:e2b-it-qat,0.984,1.56 GB / 1.56 GB,"ctx: 6144, pred: -2, temp: 1.0",m1_ctx6k_temp10_predneg2
2,gemma4:e2b-it-qat,0.987,1.56 GB / 1.56 GB,"ctx: 6144, pred: -2, temp: 0.4",m1_ctx6k_temp04_predneg2
3,gemma4:e2b-it-qat,0.987,1.56 GB / 1.56 GB,"ctx: 6144, pred: -2, temp: 0.1",m1_ctx6k_temp01_predneg2
4,gemma4:e2b-it-qat,0.987,1.56 GB / 1.56 GB,"ctx: 6144, pred: -2, temp: 0.0",m1_ctx6k_temp00_predneg2
5,gemma4:e2b-it-qat,0.987,1.56 GB / 1.56 GB,"ctx: 6144, pred: -2, temp: 0.3",m1_ctx6k_temp03_predneg2
6,gemma4:e2b-it-qat,1.025,1.58 GB / 1.44 GB,"ctx: 4096, pred: -2, temp: 0.3",m1_ctx4k_temp03_predneg2
7,gemma4:e2b-it-qat,1.025,1.58 GB / 1.44 GB,"ctx: 4096, pred: -2, temp: 0.1",m1_ctx4k_temp01_predneg2
8,gemma4:e2b-it-qat,1.025,1.58 GB / 1.44 GB,"ctx: 4096, pred: -2, temp: 1.0",m1_ctx4k_temp10_predneg2
9,gemma4:e2b-it-qat,1.025,1.58 GB / 1.44 GB,"ctx: 4096, pred: -2, temp: 0.4",m1_ctx4k_temp04_predneg2


In [5]:
display(
    non_100_gpu[
        [
            ModelStatsDisplayCols.MODEL,
            ModelStatsDisplayCols.GPU_USAGE,
            ModelStatsDisplayCols.GPU_INFO,
            ModelStatsDisplayCols.OPTIONS,
            "Run Name",
        ]
    ]
)

,Model,GPU Usage,GPU Info,Options,Run Name
0,gemma4:e2b-it-qat,0.984,1.56 GB / 1.56 GB,"ctx: 6144, pred: -2, temp: 0.2",m1_ctx6k_temp02_predneg2
1,gemma4:e2b-it-qat,0.984,1.56 GB / 1.56 GB,"ctx: 6144, pred: -2, temp: 1.0",m1_ctx6k_temp10_predneg2
2,gemma4:e2b-it-qat,0.987,1.56 GB / 1.56 GB,"ctx: 6144, pred: -2, temp: 0.4",m1_ctx6k_temp04_predneg2
3,gemma4:e2b-it-qat,0.987,1.56 GB / 1.56 GB,"ctx: 6144, pred: -2, temp: 0.1",m1_ctx6k_temp01_predneg2
4,gemma4:e2b-it-qat,0.987,1.56 GB / 1.56 GB,"ctx: 6144, pred: -2, temp: 0.0",m1_ctx6k_temp00_predneg2
5,gemma4:e2b-it-qat,0.987,1.56 GB / 1.56 GB,"ctx: 6144, pred: -2, temp: 0.3",m1_ctx6k_temp03_predneg2
6,gemma4:e2b-it-qat,1.025,1.58 GB / 1.44 GB,"ctx: 4096, pred: -2, temp: 0.3",m1_ctx4k_temp03_predneg2
7,gemma4:e2b-it-qat,1.025,1.58 GB / 1.44 GB,"ctx: 4096, pred: -2, temp: 0.1",m1_ctx4k_temp01_predneg2
8,gemma4:e2b-it-qat,1.025,1.58 GB / 1.44 GB,"ctx: 4096, pred: -2, temp: 1.0",m1_ctx4k_temp10_predneg2
9,gemma4:e2b-it-qat,1.025,1.58 GB / 1.44 GB,"ctx: 4096, pred: -2, temp: 0.4",m1_ctx4k_temp04_predneg2


## Visualizations

In [ ]:
# 1. Total Time (s) and Load Time (s)
fig, axes = plt.subplots(2, 1, figsize=(14, 12), sharex=True)

sns.barplot(
    ax=axes[0],
    data=df_all,
    x=ModelStatsDisplayCols.MODEL,
    y=ModelStatsDisplayCols.TOTAL_TIME,
    hue="Run Name",
    order=unique_models,
)
axes[0].set_title("Total Time (s) by Model and Run")
axes[0].set_ylabel("Seconds")
axes[0].legend(title="Run Config")

sns.barplot(
    ax=axes[1],
    data=df_all,
    x=ModelStatsDisplayCols.MODEL,
    y=ModelStatsDisplayCols.LOAD_TIME,
    hue="Run Name",
    order=unique_models,
)
axes[1].set_title("Load Time (s) by Model and Run")
axes[1].set_ylabel("Seconds")
axes[1].set_xlabel("Model")
axes[1].legend(title="Run Config")

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# 2. Prompt Tokens and Gen Tokens
fig, axes = plt.subplots(2, 1, figsize=(14, 12), sharex=True)

sns.barplot(
    ax=axes[0],
    data=df_all,
    x=ModelStatsDisplayCols.MODEL,
    y=ModelStatsDisplayCols.PROMPT_TOKENS,
    hue="Run Name",
    order=unique_models,
)
axes[0].set_title("Prompt Tokens by Model and Run")
axes[0].set_ylabel("Tokens")
axes[0].legend(title="Run Config")

sns.barplot(
    ax=axes[1],
    data=df_all,
    x=ModelStatsDisplayCols.MODEL,
    y=ModelStatsDisplayCols.GEN_TOKENS,
    hue="Run Name",
    order=unique_models,
)
axes[1].set_title("Generated Tokens by Model and Run")
axes[1].set_ylabel("Tokens")
axes[1].set_xlabel("Model")
axes[1].legend(title="Run Config")

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# 3. Gen Speed (t/s) and GPU Usage
fig, axes = plt.subplots(2, 1, figsize=(14, 12), sharex=True)

sns.barplot(
    ax=axes[0],
    data=df_all,
    x=ModelStatsDisplayCols.MODEL,
    y=ModelStatsDisplayCols.GEN_SPEED,
    hue="Run Name",
    order=unique_models,
)
axes[0].set_title("Generation Speed by Model and Run")
axes[0].set_ylabel("Tokens/Second")
axes[0].legend(title="Run Config")

sns.barplot(
    ax=axes[1],
    data=df_all,
    x=ModelStatsDisplayCols.MODEL,
    y=ModelStatsDisplayCols.GPU_USAGE,
    hue="Run Name",
    order=unique_models,
)
axes[1].set_title("GPU Usage by Model and Run")
axes[1].set_ylabel("Usage %")
axes[1].set_xlabel("Model")
axes[1].legend(title="Run Config")

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# 4. Response Words
plt.figure(figsize=(14, 7))
sns.barplot(
    data=df_all,
    x=ModelStatsDisplayCols.MODEL,
    y=ModelStatsDisplayCols.RESPONSE_WORDS,
    hue="Run Name",
    order=unique_models,
)
plt.title("Response Words by Model and Run")
plt.ylabel("Word Count")
plt.xlabel("Model")
plt.legend(title="Run Config")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [6]:
# Filter models with bad performance
df_filtered = df_all[
    (df_all[ModelStatsDisplayCols.GPU_USAGE] == 1.0) & (df_all[ModelStatsDisplayCols.GEN_SPEED] > 0)
].copy()

view_cols = [
    ModelStatsDisplayCols.MODEL,
    ModelStatsDisplayCols.OPTIONS,
    ModelStatsDisplayCols.TOTAL_TIME,
    ModelStatsDisplayCols.GEN_SPEED,
    ModelStatsDisplayCols.GEN_TOKENS,
    ModelStatsDisplayCols.RESPONSE_WORDS,
]
table_df = (
    df_filtered[view_cols]
    .sort_values([ModelStatsDisplayCols.GEN_TOKENS, ModelStatsDisplayCols.RESPONSE_WORDS], ascending=False)
    .reset_index(drop=True)
)
styled_df = table_df.style.background_gradient(
    cmap="RdYlGn",
    subset=[ModelStatsDisplayCols.GEN_TOKENS, ModelStatsDisplayCols.RESPONSE_WORDS, ModelStatsDisplayCols.GEN_SPEED],
).background_gradient(cmap="RdYlGn_r", subset=[ModelStatsDisplayCols.TOTAL_TIME])
display(styled_df)

,Model,Options,Total Time (s),Gen Speed (t/s),Gen Tokens,Response Words
0,gemma4:e2b-it-qat,"ctx: 14336, pred: 6144, temp: 1.0",81.306000,48.162000,3549,1384
1,gemma4:e2b-it-qat,"ctx: 12288, pred: 4096, temp: 1.0",81.394000,47.722000,3516,1436
2,gemma4:e2b-it-qat,"ctx: 16384, pred: 8192, temp: 0.1",80.533000,47.522000,3459,1464
3,gemma4:e2b-it-qat,"ctx: 16384, pred: 4096, temp: 0.3",80.067000,47.536000,3439,1355
4,gemma4:e2b-it-qat,"ctx: 12288, pred: -1, temp: 0.1",78.160000,48.121000,3394,1430
5,gemma4:e2b-it-qat,"ctx: 12288, pred: 8192, temp: 0.2",77.186000,48.720000,3388,1582
6,gemma4:e2b-it-qat,"ctx: 12288, pred: 7168, temp: 0.1",78.237000,47.938000,3385,1553
7,gemma4:e2b-it-qat,"ctx: 12288, pred: 6144, temp: 0.1",76.731000,48.119000,3325,1392
8,gemma4:e2b-it-qat,"ctx: 12288, pred: 6144, temp: 0.2",75.279000,48.493000,3282,1446
9,gemma4:e2b-it-qat,"ctx: 12288, pred: 8192, temp: 0.1",75.399000,48.339000,3277,1386


# Top one based on MinMaxScaler

In [7]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

df = table_df.copy()

scaler = MinMaxScaler()
metrics = [
    ModelStatsDisplayCols.TOTAL_TIME,
    ModelStatsDisplayCols.GEN_SPEED,
    ModelStatsDisplayCols.GEN_TOKENS,
    ModelStatsDisplayCols.RESPONSE_WORDS,
]
df_norm = pd.DataFrame(scaler.fit_transform(df[metrics]), columns=metrics)
df_norm[ModelStatsDisplayCols.TOTAL_TIME] = 1 - df_norm[ModelStatsDisplayCols.TOTAL_TIME]

weights = {
    ModelStatsDisplayCols.TOTAL_TIME: 0.20,
    ModelStatsDisplayCols.GEN_SPEED: 0.10,
    ModelStatsDisplayCols.GEN_TOKENS: 0.30,
    ModelStatsDisplayCols.RESPONSE_WORDS: 0.40,
}

CONST_SCALER_SCORE = "Scaler Score"

df[CONST_SCALER_SCORE] = (
    df_norm[ModelStatsDisplayCols.TOTAL_TIME] * weights[ModelStatsDisplayCols.TOTAL_TIME]
    + df_norm[ModelStatsDisplayCols.GEN_SPEED] * weights[ModelStatsDisplayCols.GEN_SPEED]
    + df_norm[ModelStatsDisplayCols.GEN_TOKENS] * weights[ModelStatsDisplayCols.GEN_TOKENS]
    + df_norm[ModelStatsDisplayCols.RESPONSE_WORDS] * weights[ModelStatsDisplayCols.RESPONSE_WORDS]
)

df_scaler = df.sort_values(by=CONST_SCALER_SCORE, ascending=False).reset_index(drop=True)
df_scaler[[ModelStatsDisplayCols.MODEL, ModelStatsDisplayCols.OPTIONS, CONST_SCALER_SCORE,            
ModelStatsDisplayCols.TOTAL_TIME,
            ModelStatsDisplayCols.GEN_SPEED,
            ModelStatsDisplayCols.GEN_TOKENS,
            ModelStatsDisplayCols.RESPONSE_WORDS,]].head()

,Model,Options,Scaler Score,Total Time (s),Gen Speed (t/s),Gen Tokens,Response Words
0,gemma4:e2b-it-qat,"ctx: 12288, pred: 8192, temp: 0.2",0.778995,77.186,48.720,3388,1582
1,gemma4:e2b-it-qat,"ctx: 12288, pred: 7168, temp: 0.1",0.745625,78.237,47.938,3385,1553
2,gemma4:e2b-it-qat,"ctx: 12288, pred: -2, temp: 0.2",0.737932,74.658,47.929,3211,1552
3,gemma4:e2b-it-qat,"ctx: 12288, pred: 6144, temp: 0.2",0.732791,75.279,48.493,3282,1446
4,gemma4:e2b-it-qat,"ctx: 12288, pred: -1, temp: 0.1",0.720863,78.160,48.121,3394,1430


# Top ones based on TOPSIS

In [35]:
import skcriteria as skc
from skcriteria.madm import similarity
from skcriteria.preprocessing import scalers

df = table_df.copy()

# Set objectives: min (-1), max (1), max (1), max (1)
matrix = skc.mkdm(
    df[
        [
            ModelStatsDisplayCols.TOTAL_TIME,
            ModelStatsDisplayCols.GEN_SPEED,
            ModelStatsDisplayCols.GEN_TOKENS,
            ModelStatsDisplayCols.RESPONSE_WORDS,
        ]
    ].values,
    objectives=[-1, 1, 1, 1],
    alternatives=df[ModelStatsDisplayCols.MODEL].values,
    criteria=["Time", "Speed", "Tokens", "Words"],
)

# Apply TOPSIS
scaler = scalers.VectorScaler(target="matrix")
decider = similarity.TOPSIS()
decision = decider.evaluate(scaler.transform(matrix))

CONST_TOPSIS_RANK = "TOPSIS Rank"
CONST_TOPSIS_SCORE = "TOPSIS Score"

# 1. Extract the rankings from the decision object (1 is the best)
df[CONST_TOPSIS_RANK] = decision.rank_

# 2. Extract the actual similarity score (how close it is to the "Ideal" model)
# Higher score is better (ranges from 0 to 1)
df[CONST_TOPSIS_SCORE] = decision.e_.similarity

# 3. Sort your dataframe so the #1 ranked model is at the top
df_topsis = df.sort_values(by=CONST_TOPSIS_RANK)

# 4. Use Jupyter's rich display instead of print() for a beautiful HTML table
display(
    df_topsis[
        [
            CONST_TOPSIS_RANK,
            CONST_TOPSIS_SCORE,
            ModelStatsDisplayCols.MODEL,
            ModelStatsDisplayCols.OPTIONS,
            ModelStatsDisplayCols.TOTAL_TIME,
            ModelStatsDisplayCols.GEN_SPEED,
            ModelStatsDisplayCols.GEN_TOKENS,
            ModelStatsDisplayCols.RESPONSE_WORDS,
        ]
    ].head()
)

/home/kirill/prj/gh/job-hunt/tools/py/tailor_cv_eval/.venv/lib/python3.12/site-packages/skcriteria/agg/similarity.py:124: UserWarning: Although TOPSIS can operate with minimization objectives, this is not recommended. Consider reversing the weights for these cases.
  warnings.warn(


,TOPSIS Rank,TOPSIS Score,Model,Options,Total Time (s),Gen Speed (t/s),Gen Tokens,Response Words
14,1,0.688460,gemma4:e2b-it-qat,"ctx: 12288, pred: -2, temp: 0.2",74.658,47.929,3211,1552
5,2,0.687072,gemma4:e2b-it-qat,"ctx: 12288, pred: 8192, temp: 0.2",77.186,48.720,3388,1582
26,3,0.685057,gemma4:e2b-it-qat,"ctx: 12288, pred: 6144, temp: 0.3",73.382,47.461,3118,1512
24,4,0.682438,gemma4:e2b-it-qat,"ctx: 16384, pred: -1, temp: 0.2",72.681,48.135,3130,1464
6,5,0.679141,gemma4:e2b-it-qat,"ctx: 12288, pred: 7168, temp: 0.1",78.237,47.938,3385,1553


# Best models

In [37]:
result = pd.merge(df_scaler, df_topsis, on=            [ModelStatsDisplayCols.MODEL,
            ModelStatsDisplayCols.OPTIONS,],)[[ModelStatsDisplayCols.MODEL,
            ModelStatsDisplayCols.OPTIONS,CONST_TOPSIS_RANK, CONST_TOPSIS_SCORE, CONST_SCALER_SCORE, ]]

CONST_COMBINED_RANK = "Combined Rank"
CONST_COMBINED_SCORE = "Combined Score"

df_combined = result.copy()

matrix = skc.mkdm(
    matrix=df_combined[[CONST_TOPSIS_SCORE, CONST_SCALER_SCORE]].values,
    objectives=[1, 1],  
    criteria=[CONST_TOPSIS_SCORE, CONST_SCALER_SCORE],
)

scaler = scalers.VectorScaler(target="matrix")
decider = similarity.TOPSIS()
decision = decider.evaluate(scaler.transform(matrix))

df_combined[CONST_COMBINED_RANK] = decision.rank_
df_combined[CONST_COMBINED_SCORE] = decision.e_.similarity

df_final_topsis = df_combined.sort_values(by=CONST_COMBINED_RANK)
styled_df = df_final_topsis.head(10).style.background_gradient(
    cmap="RdYlGn",
    subset=[CONST_TOPSIS_SCORE, CONST_SCALER_SCORE, CONST_COMBINED_SCORE],
).background_gradient(cmap="RdYlGn_r", subset=[CONST_COMBINED_RANK, CONST_TOPSIS_RANK])
display(styled_df)


,Model,Options,TOPSIS Rank,TOPSIS Score,Scaler Score,Combined Rank,Combined Score
0,gemma4:e2b-it-qat,"ctx: 12288, pred: 8192, temp: 0.2",2,0.687072,0.778995,1,0.997902
1,gemma4:e2b-it-qat,"ctx: 12288, pred: 7168, temp: 0.1",5,0.679141,0.745625,2,0.948184
2,gemma4:e2b-it-qat,"ctx: 12288, pred: -2, temp: 0.2",1,0.688460,0.737932,3,0.939142
3,gemma4:e2b-it-qat,"ctx: 12288, pred: 6144, temp: 0.2",8,0.674791,0.732791,4,0.927993
5,gemma4:e2b-it-qat,"ctx: 16384, pred: -1, temp: 0.2",4,0.682438,0.719151,5,0.910904
4,gemma4:e2b-it-qat,"ctx: 12288, pred: -1, temp: 0.1",22,0.663831,0.720863,6,0.905432
11,gemma4:e2b-it-qat,"ctx: 12288, pred: 6144, temp: 0.3",3,0.685057,0.708558,7,0.896092
8,gemma4:e2b-it-qat,"ctx: 12288, pred: 8192, temp: 0.1",20,0.664286,0.712253,8,0.893882
9,gemma4:e2b-it-qat,"ctx: 16384, pred: 8192, temp: 0.1",26,0.660644,0.711649,9,0.890925
10,gemma4:e2b-it-qat,"ctx: 12288, pred: 7168, temp: 0.2",18,0.664997,0.709794,10,0.890860


# Summary

This summarize results of 420 combinations of next settings:
```python
temperatures = [0.0, 0.1, 0.2, 0.3, 0.4, 1.0]
num_ctx_values = list(range(4096, 16384 + 1, 2048))
num_predict_values = [-2, -1] + list(range(1024, 8192 + 1, 1024))
```

Here are best options to try:
- for `num_ctx=16384` use `num_predict: -1, temperature: 0.2`
- for `num_ctx=12288` use `num_predict: 8192, temperature: 0.2`
- for `num_ctx=12288` use `num_predict: 7168, temperature: 0.1`
